# 5. Forecasting Model Comparison

Fit three comparable forecasters per selected store on a time-respecting split, and compare RMSE/MAE on a held-out 6-week test window. Corresponds to step 5 of the workflow in `CLAUDE.md`.

**Methods compared:**
1. **Seasonal-naive baseline** - repeats the last observed weekly cycle.
2. **SARIMA** (`statsmodels`) - small AIC-based grid search over `(p,d,q)(P,D,Q,s)`.
3. **Lag-feature gradient boosting** (`scikit-learn`) - lag/rolling-mean features of Sales plus DayOfWeek/Promo/StateHoliday/SchoolHoliday, forecast recursively.

In [ ]:
import sys, json, warnings
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from data import load_merged
from timeseries import build_store_series
from models import SeasonalNaiveForecaster, SarimaForecaster, LagFeatureMLForecaster
from evaluate import run_comparison, test_predictions

warnings.filterwarnings("ignore")

OUTPUTS = Path.cwd().parent / "outputs"
(OUTPUTS / "tables").mkdir(parents=True, exist_ok=True)
with open(OUTPUTS / "selected_stores.json") as f:
    store_ids = json.load(f)

merged = load_merged()

In [ ]:
forecaster_factories = {
    "seasonal_naive": lambda period: SeasonalNaiveForecaster(period),
    "sarima": lambda period: SarimaForecaster(period),
    "lag_ml": lambda period: LagFeatureMLForecaster(period, model="gbrt"),
}

## Run the time-respecting comparison per store

In [ ]:
all_results = []
for sid in store_ids:
    series = build_store_series(merged, sid)
    print(f"Store {sid}: seasonal period = {series.seasonal_period} "
          f"trading days/week, {len(series.frame)} open days")
    all_results.append(run_comparison(series, forecaster_factories))

results = pd.concat(all_results, ignore_index=True)
results.to_csv(OUTPUTS / "tables" / "model_comparison.csv", index=False)
results

## Example forecasts (actual vs. predicted, test window)

Saved for all 6 stores so any one of them can be plotted in the results summary as a concrete, non-technical illustration of what the RMSE/MAE numbers above actually mean.

In [ ]:
example_predictions = []
for sid in store_ids:
    series = build_store_series(merged, sid)
    example_predictions.append(test_predictions(series, forecaster_factories))

example_predictions = pd.concat(example_predictions, ignore_index=True)
example_predictions.to_csv(OUTPUTS / "tables" / "test_predictions.csv", index=False)
example_predictions.head()

## Test-set summary (final reported metric)

In [ ]:
test_summary = (
    results.query("split == 'test'")
    .pivot(index="store", columns="method", values="rmse")
    .round(1)
)
test_summary